# Exercise: your first GPU job on Nautilus

A complete, hands-on walkthrough for Shi Lab students. You will:

1. verify your cluster access,
2. see which GPUs exist right now,
3. launch a debug pod with one GPU,
4. run a sanity script on the GPU and read its energy counter,
5. convert the same work into a proper batch **Job**,
6. clean everything up.

**Prerequisites:** `kubectl` + `kubelogin` installed and your `~/.kube/config` set up (step 0 of the README). Every cell runs on *your laptop* — `kubectl` talks to the cluster for you.

> ⏱ ~20 minutes. Cost: one small free-pool GPU while you work — **run the cleanup cells when done.**

## 1 · Verify access

The context should be `nautilus` and the pod list should return without errors
(an empty list is fine). If auth is stale: `kubectl oidc-login clean` and retry.

In [ ]:
!kubectl config current-context
!kubectl get pods

## 2 · What GPUs exist right now?

Nautilus nodes are labeled with their GPU product. This is the live truth — check it
(together with the [NRP resources page](https://portal.nrp-nautilus.io/resources))
whenever you wonder what to request.

In [ ]:
!kubectl get nodes -L nvidia.com/gpu.product | awk '{print $NF}' | sort | uniq -c | sort -rn | head -15

## 3 · Write the pod spec — from this notebook

The cell below writes `lab-tutorial-pod.yaml`. Read it before running: it asks for **one**
free-pool GPU (several acceptable products, so the scheduler picks any free card),
sets `requests` = `limits`, installs deps, prints `READY_FOR_CODE`, and sleeps so we
can drive it by hand.

> ⚠ **Lab rule:** a sleeping pod is for *debug sessions only* — never for reserving a
> GPU to "train later". Real runs go in a Job (part 6). Delete this pod the same day.

In [ ]:
%%writefile lab-tutorial-pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: lab-tutorial-pod   # rename to <yourname>-tutorial if others may run this too
spec:
  restartPolicy: Never
  affinity:
    nodeAffinity:
      requiredDuringSchedulingIgnoredDuringExecution:
        nodeSelectorTerms:
          - matchExpressions:
              - key: nvidia.com/gpu.product
                operator: In
                values:
                  - NVIDIA-GeForce-RTX-2080-Ti
                  - Tesla-T4
                  - NVIDIA-A10
                  - NVIDIA-GeForce-RTX-3090
                  - NVIDIA-A40
                  - NVIDIA-L40
  containers:
    - name: main
      image: pytorch/pytorch:2.4.0-cuda12.1-cudnn9-runtime
      command: ["/bin/bash", "-lc"]
      args:
        - |
          set -euo pipefail
          echo "=== node: $(hostname) ==="; nvidia-smi -L
          pip install -q nvidia-ml-py
          mkdir -p /workspace /output
          echo READY_FOR_CODE
          sleep infinity
      resources:
        limits:   {nvidia.com/gpu: "1", cpu: "2", memory: 8Gi}
        requests: {nvidia.com/gpu: "1", cpu: "2", memory: 8Gi}
      volumeMounts:
        - {name: workspace, mountPath: /workspace}
        - {name: output, mountPath: /output}
  volumes:
    - {name: workspace, emptyDir: {}}
    - {name: output, emptyDir: {}}

## 4 · Launch it and wait until it's ready

`apply` submits the pod; the loop below polls until the container prints
`READY_FOR_CODE`. Scheduling usually takes 1–5 minutes. If it stays `Pending` for
long, diagnose with the `describe` cell further down.

In [ ]:
!kubectl apply -f lab-tutorial-pod.yaml

In [ ]:
import subprocess, time
for i in range(60):
    logs = subprocess.run(['kubectl', 'logs', 'lab-tutorial-pod'],
                          capture_output=True, text=True).stdout
    if 'READY_FOR_CODE' in logs:
        print(logs); print('--- pod is ready ---'); break
    phase = subprocess.run(['kubectl', 'get', 'pod', 'lab-tutorial-pod',
                            '-o', 'jsonpath={.status.phase}'],
                           capture_output=True, text=True).stdout
    print(f'[{i*10:4d}s] phase={phase} — waiting…')
    time.sleep(10)
else:
    print('Not ready after 10 min — run the describe cell below to see why.')

If something looks wrong, `describe` shows the scheduler's events — the single most
useful debugging command on Kubernetes:

In [ ]:
!kubectl describe pod lab-tutorial-pod | tail -15

## 5 · Run a sanity script on the GPU

The cell writes `hello_gpu.py` locally, copies it into the pod, and runs it. It
reports which card you got, whether you have it **exclusively**, and whether the
**NVML energy counter** works — the counter behind the lab's energy measurements.

In [ ]:
%%writefile hello_gpu.py
import pynvml
pynvml.nvmlInit()
h = pynvml.nvmlDeviceGetHandleByIndex(0)
mem = pynvml.nvmlDeviceGetMemoryInfo(h)
procs = pynvml.nvmlDeviceGetComputeRunningProcesses(h)
print('GPU       :', pynvml.nvmlDeviceGetName(h))
print(f'VRAM      : {mem.total/2**30:.1f} GiB total, {mem.used/2**30:.1f} used')
print(f'power now : {pynvml.nvmlDeviceGetPowerUsage(h)/1000:.1f} W')
print('exclusive :', len(procs) == 0, f'({len(procs)} other compute processes)')
try:
    e = pynvml.nvmlDeviceGetTotalEnergyConsumption(h)
    print(f'energy ctr: OK ({e/1000:.0f} J since driver load)')
except pynvml.NVMLError as ex:
    print('energy ctr: NOT available ->', ex, '(measure energy as integral of power)')
print('HELLO_GPU_DONE')

In [ ]:
!kubectl cp hello_gpu.py lab-tutorial-pod:/workspace/hello_gpu.py
!kubectl exec lab-tutorial-pod -- python /workspace/hello_gpu.py

**Checkpoint — you just did the full debug-pod cycle:** submit → wait → copy code in
→ execute → read results. For anything that runs longer than a minute, launch it
detached so a dropped connection can't kill it:

```bash
kubectl exec lab-tutorial-pod -- bash -c 'nohup python /workspace/long.py > /output/run.log 2>&1 & echo started'
kubectl exec lab-tutorial-pod -- tail -5 /output/run.log   # check whenever
```

## 6 · Clean up the pod — then do it the *right* way: a Job

Delete the debug pod (lab rule: same day, always):

In [ ]:
!kubectl delete pod lab-tutorial-pod

Now the same work as a **batch Job** — the lab default for real runs: the command
lives *in the YAML*, runs to completion, retries once on failure, and is the record
of exactly what ran. No sleeping, no manual steps.

In [ ]:
%%writefile lab-tutorial-job.yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: lab-tutorial-job   # rename to <yourname>-<experiment>
spec:
  backoffLimit: 1
  template:
    spec:
      restartPolicy: Never
      affinity:
        nodeAffinity:
          requiredDuringSchedulingIgnoredDuringExecution:
            nodeSelectorTerms:
              - matchExpressions:
                  - key: nvidia.com/gpu.product
                    operator: In
                    values: [NVIDIA-GeForce-RTX-2080-Ti, Tesla-T4, NVIDIA-A10, NVIDIA-GeForce-RTX-3090, NVIDIA-A40, NVIDIA-L40]
      containers:
        - name: main
          image: pytorch/pytorch:2.4.0-cuda12.1-cudnn9-runtime
          command: ["/bin/bash", "-lc"]
          args:
            - |
              set -euo pipefail
              pip install -q nvidia-ml-py
              python - <<'PY'
              import pynvml
              pynvml.nvmlInit()
              h = pynvml.nvmlDeviceGetHandleByIndex(0)
              print('GPU:', pynvml.nvmlDeviceGetName(h))
              print('JOB_DONE')
              PY
          resources:
            limits:   {nvidia.com/gpu: "1", cpu: "2", memory: 8Gi}
            requests: {nvidia.com/gpu: "1", cpu: "2", memory: 8Gi}

In [ ]:
!kubectl apply -f lab-tutorial-job.yaml

In [ ]:
import subprocess, time
for i in range(60):
    st = subprocess.run(['kubectl', 'get', 'job', 'lab-tutorial-job',
                         '-o', 'jsonpath={.status.succeeded}'],
                        capture_output=True, text=True).stdout
    if st == '1':
        print('--- job succeeded; logs: ---')
        print(subprocess.run(['kubectl', 'logs', 'job/lab-tutorial-job'],
                             capture_output=True, text=True).stdout)
        break
    print(f'[{i*10:4d}s] waiting for the job…'); time.sleep(10)
else:
    print('Still not done — kubectl describe pod / logs to investigate.')

## 7 · Final cleanup + your checklist

Leave the namespace as you found it:

In [ ]:
!kubectl delete job lab-tutorial-job
!kubectl get pods    # should not show anything of yours

**You now know the full loop.** Before your first real run, go back to the README:

- Step 1 — the lab workflow (local debug → smoke test → full run → cleanup)
- Step 6 — put checkpoints on the **PVC** (this exercise used emptyDir: everything died with the pod, on purpose)
- Step 7 — secrets for HF/W&B tokens
- Step 9 — the errors table, *before* you hit them
- Step 10 — the lab rules checklist (tracker, GitHub commit, smoke test, cleanup)